# Wall-breaker benchmark visualisation

Reads `results/summary.csv` produced by `run_all.py` and renders:

1. A feasibility / L2-distance scatter, one point per (method, fixture, z), with the manuscript baseline marked for reference.
2. The winning corrected slice next to the input (warped grid + per-cell `min(T1,T2)` heatmap).

In [ ]:
import os, sys
sys.path.insert(0, '.')
sys.path.insert(0, os.path.abspath('../../..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

import harness as H
from dvfopt.jacobian.triangle_sign import _triangle_areas_2d

df = pd.read_csv('results/summary.csv')
print(df.shape, 'results')
df.head(3)

In [ ]:
# Feasibility pass-rate per method, with median L2 cost.
summary = df.groupby('method').agg(
    n=('feasible_2tri', 'size'),
    feasible_pct=('feasible_2tri', lambda s: 100 * s.mean()),
    L2_median=('l2_delta', 'median'),
    tri_min_median=('final_tri_min', 'median'),
    wall_median=('wall_s', 'median'),
).sort_values('feasible_pct', ascending=False)
summary.round(2)

In [ ]:
# Scatter: x = L2, y = min(T1, T2). Threshold line at 0.01.
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
for m, g in df[df['error'].isna()].groupby('method'):
    feas = g['feasible_2tri']
    ax.scatter(g['l2_delta'], g['final_tri_min'], s=70,
               label=f"{m}  ({100*feas.mean():.0f}% feas)",
               marker='o' if feas.all() else 'x' if not feas.any() else 's',
               alpha=0.85, edgecolors='k', linewidths=0.4)
ax.axhline(0.01, color='g', ls='--', lw=1, label='threshold')
ax.axhline(0.0, color='k', ls=':', lw=0.8, label='fold-free boundary')
ax.set_xlabel(r'$\|\\phi_\\mathrm{out} - \\phi_\\mathrm{in}\\|_2$ (lower = closer to input)')
ax.set_ylabel('final min(T1, T2)  (must exceed threshold)')
ax.set_title('Wall breaker comparison: L2 cost vs final feasibility')
ax.legend(fontsize=8, loc='lower right')
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# Per-z winner table (lowest L2 among the feasible methods, per fixture).
ok = df[df['feasible_2tri'] == True].copy()  # noqa
if not ok.empty:
    winners = ok.loc[ok.groupby(['fixture', 'z'])['l2_delta'].idxmin()]
    print('Per-(fixture, z) winners (lowest L2 among feasibility-achieving methods):')
    print(winners[['fixture', 'z', 'method', 'l2_delta', 'final_tri_min', 'wall_s']].to_string(index=False))
else:
    print('No method achieved full feasibility on any fixture.')

In [ ]:
# Warped-grid panels: input vs winner output for one slice.
TARGET_Z = 12
TARGET_FIX = 'slice'

vol = H.load_volume()
phi_in = H.get_slice(vol, TARGET_Z)

row = df[(df['fixture'] == TARGET_FIX) & (df['z'] == TARGET_Z) & (df['feasible_2tri'] == True)]
if not row.empty:
    row = row.sort_values('l2_delta').iloc[0]
    fn = f'results/{row["method"]}__{row["fixture"]}__z{int(row["z"]):03d}.npy'
    if os.path.isfile(fn):
        phi_out = np.load(fn)
        T1, T2 = _triangle_areas_2d(phi_out[0], phi_out[1])
        T1i, T2i = _triangle_areas_2d(phi_in[0], phi_in[1])

        fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)
        for ax, T_in, T_use, ph, title in [(axes[0], None, np.minimum(T1i, T2i), phi_in, 'input'),
                                            (axes[1], None, np.minimum(T1, T2), phi_out,
                                             f'after {row["method"]}')]:
            vmax = max(abs(T_use.min()), abs(T_use.max()), 0.1)
            im = ax.imshow(T_use, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
            ax.set_title(f'{title}  min={T_use.min():+.4f}  n_neg={int((T_use<=0).sum())}',
                         fontsize=11)
            ax.set_xticks([]); ax.set_yticks([])
            fig.colorbar(im, ax=ax, shrink=0.85)
        fig.suptitle(f'z={TARGET_Z} {TARGET_FIX}: input -> {row["method"]}  '
                     f'(L2={row["l2_delta"]:.1f}, {row["wall_s"]:.1f}s)', fontsize=12)
        plt.show()
    else:
        print(f'no saved npy for winner: {fn}')
else:
    print(f'no feasible result for z={TARGET_Z} fixture={TARGET_FIX}')

## Reading the numbers

Three quantities matter, in this order:

1. **`feasible_2tri`** — did the method actually clear the 2-tri threshold? `True` is the only acceptable answer.
2. **`l2_delta`** — among feasibility-achieving methods, how close did it stay to the input field? Lower is better.
3. **`wall_s`** — wall clock. Useful but secondary: a method that gets the right answer in 60 s is worth more than one that gets the wrong answer in 1 s.

The winners table above selects the method with the lowest `l2_delta` among those satisfying `feasible_2tri == True` for each `(fixture, z)`. If any cell shows the same method winning across all wall slices, that's your single-method answer for the full pipeline.